# TP NovaCommerce · notebook d'audit

**DATE732 · Qualité & intégrité des données · accompagne le document `TP_DATE732_NovaCommerce_etudiants.docx`**

Avant ce notebook, faites `00_bases_python_pandas.ipynb` si pandas ne vous est pas familier.

| Partie du notebook | Mission du TP | Bloc |
|---|---|---|
| 1. Chargement · 2. Boîte à outils | toutes | — |
| 3. Inventaire des sources | Mission 1 · données critiques | Bloc 1 · Jour 1 |
| 4. Tableaux, JSON, XML : profiling, R01 à R03 | Mission 2 · audit | Bloc 1 |
| 5. Logs, IoT, GPS, PDF, texte : R04 à R08, R09, R10 | Mission 3 · audit hors tableaux | Bloc 1 |
| 6. Suivre une commande de bout en bout | Mission 4 · réconciliation | Bloc 1 |
| 7. Synthèse et export | Mission 5 · remédiation | Bloc 1 |
| 8. Journaux et lineage | Mission 6 · contrôles et traçabilité | Bloc 2 · Jour 2 |
| 9. CRM et ERP | Mission 7 · données de référence | Bloc 2 |
| 10. Dérive de schéma et moniteurs | Mission 8 · contrat et observabilité | Bloc 2 |
| 11. Indicateurs | Mission 9 · décisionnel | Bloc 2 |

**Règles du jeu**
- Exécutez les cellules dans l'ordre. Les parties 1 à 5 sont prêtes à exécuter ; les cellules **À vous** sont à compléter.
- Ne modifiez jamais le dossier `donnees/`. Une valeur rare n'est pas forcément fausse ; une valeur bien formée n'est pas forcément juste.
- Pour chaque résultat, notez : la règle, ce qui est compté (lignes ? clés ?), le dénominateur, l'impact métier.
- Les fonctions donnent des **mesures**. L'interprétation (erreur certaine, suspicion, cas légitime) est votre travail.

In [ ]:
import json, re
import xml.etree.ElementTree as ET
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 160)
D = Path('donnees')   # dossier des données brutes, à côté du notebook

## 1. Chargement des quatorze sources

Tout est lu **en texte** pour ne rien « corriger » sans le vouloir (zéros initiaux, dates, codes).

In [ ]:
lire = lambda nom: pd.read_csv(D / nom, dtype=str, keep_default_na=False)
clients, erp, produits = lire('clients_crm.csv'), lire('clients_facturation.csv'), lire('produits.csv')
refs = pd.read_csv(D / 'referentiels.csv', dtype=str)
print('clients CRM :', len(clients), '· clients ERP :', len(erp), '· produits :', len(produits), '· codes de référence :', len(refs))

In [ ]:
# JSON : une ligne par commande, et une ligne par article commandé
with open(D / 'commandes_api.json', encoding='utf-8') as f:
    brut = json.load(f)
commandes = pd.json_normalize(brut)
lignes = pd.json_normalize(brut, record_path='lignes', meta=['commande_id'])
print('commandes :', len(commandes), '· lignes de commande :', len(lignes))
print('colonnes :', list(commandes.columns))

In [ ]:
# XML : une ligne par facture
racine = ET.parse(D / 'factures.xml').getroot()
factures = pd.DataFrame([{
    'numero': f.get('numero'), 'commande_id': f.findtext('commande_id'), 'client_id': f.findtext('client_id'),
    'date_emission': f.findtext('date_emission'), 'total_ht': f.findtext('total_ht'), 'total_tva': f.findtext('total_tva'),
    'total_ttc': f.findtext('total_ttc'), 'devise': f.findtext('devise'), 'nb_lignes': len(f.find('lignes')),
} for f in racine.findall('facture')])
print('factures :', len(factures))

In [ ]:
# Logs : un événement de paiement par ligne. On compte aussi les lignes qui ne correspondent pas au motif.
motif = re.compile(r'^(?P<recu>\S+) (?P<niveau>\w+) psp-gateway event_id=(?P<event_id>\S+) '
                   r'event_time=(?P<event_time>.+?) type=(?P<type>\S+) commande_id=(?P<commande_id>\S+) '
                   r'montant=(?P<montant>\S*) devise=(?P<devise>\S+) moyen=(?P<moyen>\S+)$')
evts, autres = [], []
with open(D / 'paiements.log', encoding='utf-8') as f:
    for ligne in f:
        m = motif.match(ligne.strip())
        (evts.append(m.groupdict()) if m else autres.append(ligne.strip()))
paiements = pd.DataFrame(evts)
print('événements de paiement :', len(paiements), '· autres lignes :', len(autres), '→ exemple :', autres[:1])

In [ ]:
# Texte libre, série temporelle IoT, géographique, documents, et les fichiers de la Partie II
avis, capteurs, livraisons, pdf = lire('avis_clients.csv'), lire('capteurs_froid.csv'), lire('livraisons_gps.csv'), lire('factures_pdf_index.csv')
journaux, lineage, kpi = lire('journaux_traitements.csv'), lire('flux_lineage.csv'), lire('kpi_bruts_a_verifier.csv')
print('avis :', len(avis), '· mesures :', len(capteurs), '· livraisons :', len(livraisons), '· PDF indexés :', len(pdf))
print('exécutions journalisées :', len(journaux), '· flux déclarés :', len(lineage), '· KPI publiés :', len(kpi))

## 2. Boîte à outils

Des fonctions génériques pour **mesurer**. Elles ne disent pas ce qui est faux : elles comptent.

| Fonction | Ce qu'elle mesure |
|---|---|
| `profil(df)` | vides, valeurs distinctes, valeurs les plus fréquentes par colonne |
| `resultat(regle, ...)` | enregistre une règle dans la synthèse (contrôlés, en anomalie, % conformes) |
| `hors_referentiel(serie, type)` | valeurs absentes de `referentiels.csv` |
| `doublons(df, cles)` | lignes concernées **et** clés distinctes concernées |
| `format_invalide(serie, motif)` | valeurs qui ne respectent pas une expression régulière |
| `en_nombre(serie)` · `en_date(serie)` | conversion, en comptant les échecs |
| `orphelins(df, col, cible, col_cible)` | références vides et références inconnues |
| `controle_jointure(g, d, cle)` | cardinalité (1:1, 1:N…) et effet de la jointure sur le nombre de lignes |
| `valeur_figee(df, ...)` | mesures strictement constantes sur une fenêtre de temps |
| `distance_km(...)` · `distance_au_centre(df)` | distances GPS, écart au centre observé de la ville |
| `ecart_a_la_reference(serie)` | écart d'un volume quotidien à la moyenne des 7 jours précédents |
| `suivre(commande_id)` | une commande dans toutes les sources |

In [ ]:
def profil(df, top=4):
    """Une ligne par colonne : vides, % vides, valeurs distinctes, valeurs les plus fréquentes."""
    res = []
    for col in df.columns:
        s = df[col].astype(str)
        vides = s.str.strip().isin(['', 'nan', 'None', '<NA>']).sum()
        res.append({'colonne': col, 'vides': vides, '% vides': round(100 * vides / len(df), 2),
                    'distinctes': s.nunique(), 'plus fréquentes': s.value_counts().head(top).to_dict()})
    return pd.DataFrame(res)


synthese = []

def resultat(regle, description, controles, anomalies, exemples=None):
    """Enregistre une règle dans la synthèse et affiche quelques exemples."""
    n = anomalies if isinstance(anomalies, (int, np.integer)) else len(anomalies)
    taux = round(100 * (controles - n) / controles, 2) if controles else None
    synthese.append({'règle': regle, 'description': description, 'contrôlés': controles, 'en anomalie': n, '% conformes': taux})
    print(f'{regle} · {description}\n  contrôlés : {controles} · en anomalie : {n} · conformes : {taux} %')
    if exemples is not None and len(exemples):
        display(exemples.head(5))


def hors_referentiel(serie, type_ref, normaliser=False):
    """Valeurs absentes de referentiels.csv pour ce type (normaliser=True : ignore casse et espaces)."""
    codes = set(refs.loc[refs['type_referentiel'] == type_ref, 'code'])
    s = serie.fillna('').astype(str)
    if normaliser:
        s = s.str.strip().str.upper()
    hors = serie[~s.isin(codes)]
    print(f'{type_ref} : {len(hors)} valeurs hors référentiel sur {len(serie)} → {hors.value_counts(dropna=False).head(6).to_dict()}')
    return hors


def doublons(df, cles):
    """Toutes les lignes dont la clé est dupliquée. Affiche lignes ET clés distinctes."""
    cles = [cles] if isinstance(cles, str) else cles
    m = df.duplicated(subset=cles, keep=False)
    print(f'{cles} : {m.sum()} lignes concernées · {df.loc[m, cles].drop_duplicates().shape[0]} clés distinctes concernées')
    return df[m].sort_values(cles)


def format_invalide(serie, motif, vides_acceptes=False):
    """Valeurs qui ne respectent pas le motif (expression régulière ancrée au début)."""
    s = serie.fillna('').astype(str)
    ko = ~s.str.match(motif)
    if vides_acceptes:
        ko &= s.str.strip() != ''
    print(f'{ko.sum()} valeurs hors format sur {len(s)}')
    return serie[ko]


def en_nombre(serie):
    """Conversion en nombre ; affiche le nombre de valeurs non vides non convertibles."""
    n = pd.to_numeric(serie, errors='coerce')
    echecs = n.isna() & serie.notna() & (serie.astype(str).str.strip() != '')
    if echecs.any():
        print(f'{echecs.sum()} valeurs non numériques, par exemple {serie[echecs].astype(str).unique()[:4].tolist()}')
    return n


def en_date(serie, utc=True):
    """Conversion ISO 8601 (avec ou sans fuseau) ; affiche les échecs. Sans fuseau, la valeur est SUPPOSÉE en UTC."""
    d = pd.to_datetime(serie, format='ISO8601', errors='coerce', utc=utc)
    echecs = d.isna() & serie.notna() & (serie.astype(str).str.strip() != '')
    if echecs.any():
        print(f'{echecs.sum()} dates non ISO 8601, par exemple {serie[echecs].astype(str).unique()[:4].tolist()}')
    return d


def orphelins(df, col, cible, col_cible):
    """Références vides et références absentes de la table cible."""
    s = df[col]
    vide = s.isna() | (s.astype(str).str.strip() == '')
    inconnu = ~vide & ~s.isin(set(cible[col_cible]))
    print(f'{col} → {col_cible} : {vide.sum()} vides · {inconnu.sum()} inconnues · sur {len(df)}')
    return df[vide | inconnu]


def controle_jointure(gauche, droite, cle_g, cle_d=None):
    """Cardinalité d'une jointure et effet sur le nombre de lignes, AVANT de l'utiliser."""
    cle_d = cle_d or cle_g
    mg, md = gauche[cle_g].value_counts().max(), droite[cle_d].value_counts().max()
    card = f"{'1' if mg == 1 else 'N'}:{'1' if md == 1 else 'N'}"
    j = gauche.merge(droite, left_on=cle_g, right_on=cle_d, how='inner')
    print(f'cardinalité {card} (max {mg} à gauche, {md} à droite) · {len(gauche)} lignes à gauche → {len(j)} après jointure')
    return card


def valeur_figee(df, groupe, temps, valeur, fenetre='1h', min_points=12):
    """Lignes où la valeur est strictement constante sur toute la fenêtre glissante (écart-type nul)."""
    out = []
    for g, d in df.assign(_t=pd.to_datetime(df[temps]), _v=pd.to_numeric(df[valeur], errors='coerce')).sort_values('_t').groupby(groupe):
        r = d.set_index('_t')['_v'].rolling(fenetre)
        fig = (r.std() == 0) & (r.count() >= min_points)
        out.append(d[fig.values])
    res = pd.concat(out).drop(columns=['_t', '_v'])
    print(f'{len(res)} mesures dans une fenêtre de {fenetre} sans aucune variation')
    return res


def distance_km(lat1, lon1, lat2, lon2):
    """Distance à vol d'oiseau (formule de haversine), en km."""
    lat1, lon1, lat2, lon2 = (np.radians(pd.to_numeric(x, errors='coerce')) for x in (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))


def distance_au_centre(df, ville='ville', lat='latitude', lon='longitude'):
    """Distance de chaque point au centre observé de sa ville (médiane des points de la ville)."""
    d = df.assign(_lat=pd.to_numeric(df[lat], errors='coerce'), _lon=pd.to_numeric(df[lon], errors='coerce'))
    centre = d.groupby(ville)[['_lat', '_lon']].transform('median')
    return distance_km(d['_lat'], d['_lon'], centre['_lat'], centre['_lon'])


def ecart_a_la_reference(par_jour, fenetre=7):
    """Écart (%) de chaque jour à la moyenne des `fenetre` jours précédents."""
    ref = par_jour.rolling(fenetre).mean().shift(1)
    return pd.DataFrame({'valeur': par_jour, 'reference': ref.round(1), 'ecart_%': (100 * (par_jour - ref) / ref).round(1)})


def suivre(commande_id):
    """Affiche une commande dans toutes les sources."""
    vues = {'commande (JSON)': commandes[commandes['commande_id'] == commande_id].drop(columns=['lignes']),
            'factures (XML)': factures[factures['commande_id'] == commande_id],
            'paiements (logs)': paiements[paiements['commande_id'] == commande_id],
            'livraisons (GPS)': livraisons[livraisons['commande_id'] == commande_id],
            'PDF indexés': pdf[pdf['commande_id'] == commande_id]}
    for nom, v in vues.items():
        print(f'— {nom} : {len(v)} ligne(s)')
        if len(v):
            display(v)

print('boîte à outils prête')

In [ ]:
# Démonstration sur le référentiel produits
_ = doublons(produits, 'produit_id')
_ = hors_referentiel(produits['categorie'], 'CATEGORIE_PRODUIT')
_ = hors_referentiel(produits['categorie'], 'CATEGORIE_PRODUIT', normaliser=True)

## 3. Mission 1 · Inventaire des sources

Point de départ de la matrice des données critiques (L1, à remplir dans `modeles/L1_matrice_donnees_critiques.csv`).

In [ ]:
sources = {'clients_crm.csv': ('Tabulaire', clients), 'clients_facturation.csv': ('Tabulaire', erp), 'produits.csv': ('Tabulaire', produits),
           'commandes_api.json': ('Semi-structuré JSON', commandes), 'factures.xml': ('Semi-structuré XML', factures),
           'paiements.log': ('Logs', paiements), 'capteurs_froid.csv': ('Série temporelle IoT', capteurs),
           'livraisons_gps.csv': ('Géographique', livraisons), 'factures_pdf_index.csv': ('Documents', pdf), 'avis_clients.csv': ('Texte libre', avis)}
pd.DataFrame([{'source': k, 'type': t, 'lignes': len(df), 'colonnes': df.shape[1], 'champs': ', '.join(map(str, df.columns[:8])) + ('…' if df.shape[1] > 8 else '')}
              for k, (t, df) in sources.items()])

## 4. Mission 2 · Tableaux, JSON et XML

### 4.1 Profiling
Regardez surtout les valeurs fréquentes **et** les valeurs rares.

In [ ]:
profil(clients)

In [ ]:
profil(erp)

In [ ]:
profil(produits)

In [ ]:
profil(commandes.drop(columns=['lignes']))

In [ ]:
profil(factures)

**À vous** · comptez les commandes par `schema_version` et, pour chaque version, le nombre de valeurs vides par colonne. Qu'est-ce qui change ?

In [ ]:
# Votre code ici
# Piste : value_counts(), puis groupby('schema_version') et isna().sum()

### 4.2 Unicité, référentiels, formats

**À vous** · unicité de `clients.client_id` et de `commandes.commande_id`. Pour les commandes, séparez les doublons exacts des identifiants réutilisés avec un contenu différent.

In [ ]:
# Votre code ici
# Piste : doublons(...), puis pour les commandes : astype(str), groupby('commande_id') et drop_duplicates()

**À vous** · confrontez à `referentiels.csv` : statut client, statut de commande, code pays, catégorie et taux de TVA des produits.

In [ ]:
# Votre code ici
# Piste : hors_referentiel(colonne, 'STATUT_CLIENT') ; les types disponibles sont dans refs['type_referentiel'].unique()

**À vous** · contrôlez au moins trois formats du CRM : e-mail, date de création ISO 8601 (AAAA-MM-JJ), téléphone français à 10 chiffres, code postal à 5 chiffres.

In [ ]:
# Votre code ici
# Piste : format_invalide(clients['email'], r'^...$', vides_acceptes=True)

### 4.3 Intégrité, montants, dates, et règles R01 à R03

In [ ]:
# R01 · Chaque commande référence un client existant dans le CRM
ids_crm = set(clients['client_id'])
r01 = commandes[~commandes['client_id'].isin(ids_crm)]
resultat('R01', 'client_id existe dans le CRM', len(commandes), r01, r01[['commande_id', 'client_id', 'schema_version']])
# Le nombre vous surprend ? Comparez les colonnes et la version du schéma des commandes concernées.

In [ ]:
# R02 · Chaque identifiant client est unique
r02 = clients[clients['client_id'].duplicated(keep=False)].sort_values('client_id')
resultat('R02', 'client_id unique (lignes concernées)', len(clients), r02, r02[['client_id', 'prenom', 'nom', 'email']])

In [ ]:
# R03 · Sur chaque facture, TTC = HT + TVA (à 0,01 € près)
f = factures.copy()
for c in ['total_ht', 'total_tva', 'total_ttc']:
    f[c] = pd.to_numeric(f[c], errors='coerce')
r03 = f[(f['total_ttc'] - f['total_ht'] - f['total_tva']).abs() > 0.01]
resultat('R03', 'TTC = HT + TVA', len(f), r03, r03[['numero', 'total_ht', 'total_tva', 'total_ttc']])

**À vous** · expliquez R01 : séparez les commandes **sans aucun identifiant client**, celles dont l'identifiant est **sous un autre nom de champ**, et celles qui pointent vers un **client inconnu**.

In [ ]:
# Votre code ici
# Piste : client = commandes['client_id'].fillna(commandes['customerId'])

**À vous** · intégrité des produits commandés (`lignes.produit_id` → `produits`) et des factures (`factures.commande_id` → `commandes`).

In [ ]:
# Votre code ici
# Piste : orphelins(lignes, 'produit_id', produits, 'produit_id')

**À vous** · cohérence des montants : `montant_total_ht` = Σ quantité × prix unitaire HT (tolérance 0,01 €). Repérez les montants non numériques et les ordres de grandeur suspects.

In [ ]:
# Votre code ici
# Piste : agréger lignes par commande_id, puis comparer à montant_total_ht ; en_nombre signale le texte

**À vous** · cohérence temporelle : date de mise à jour antérieure à la création (CRM), date de naissance future, commande datée dans le futur, facture émise avant sa commande.

In [ ]:
# Votre code ici
# Piste : en_date(...) puis comparer deux colonnes ; pd.Timestamp('2026-11-01', tz='UTC') pour « aujourd'hui »

## 5. Mission 3 · Logs, IoT, GPS, documents, texte

### 5.1 Profiling

In [ ]:
profil(paiements)

In [ ]:
profil(capteurs)

In [ ]:
profil(livraisons, top=3)

In [ ]:
profil(pdf, top=3)

In [ ]:
profil(avis, top=3)

### 5.2 Règles R04 à R08

In [ ]:
# R04 · Aucun IBAN, téléphone ou e-mail dans un avis publié
iban = r'FR\d{2}(?:\s?\d{4}){5}\s?\d{3}'
tel = r'\b0[1-9](?:[\s.]?\d{2}){4}\b'
mail = r'[\w.+-]+@[\w-]+\.[\w.]+'
r04 = avis[avis['texte'].str.contains(f'{iban}|{tel}|{mail}', regex=True)]
resultat('R04', 'aucune donnée personnelle dans le texte', len(avis), r04, r04[['avis_id', 'texte']])

In [ ]:
# R05 · Chaque événement de paiement est unique et horodaté en UTC
dup = paiements['event_id'].duplicated(keep=False)
pas_utc = ~paiements['event_time'].str.endswith('Z')
r05 = paiements[dup | pas_utc]
print('  dont event_id en double :', dup.sum(), '· horodatages hors UTC :', pas_utc.sum())
resultat('R05', 'event_id unique et horodatage UTC', len(paiements), r05, r05[['event_id', 'event_time', 'commande_id', 'montant']])

In [ ]:
# R06 · Mesure plausible (−40 à +60 °C) et une mesure toutes les 5 minutes par capteur
c = capteurs.copy()
c['t'] = pd.to_numeric(c['temperature'], errors='coerce')
c['h'] = pd.to_datetime(c['horodatage_local'])
hors_plage = c[(c['t'] < -40) | (c['t'] > 60)]
c = c.sort_values(['capteur_id', 'h'])
c['ecart_min'] = c.groupby('capteur_id')['h'].diff().dt.total_seconds() / 60
trous = c[c['ecart_min'] > 5]
print('  valeurs hors plage :', len(hors_plage), '· écarts de plus de 5 min :', len(trous))
resultat('R06', 'plage du capteur et fréquence de 5 min', len(c), len(hors_plage) + len(trous),
         pd.concat([hors_plage, trous])[['capteur_id', 'horodatage_local', 'temperature', 'unite', 'ecart_min']])

In [ ]:
# R07 · Point de livraison en France métropolitaine (boîte englobante approximative)
g = livraisons.copy()
g['lat'] = pd.to_numeric(g['latitude'], errors='coerce')
g['lon'] = pd.to_numeric(g['longitude'], errors='coerce')
r07 = g[~g['lat'].between(41, 51.5) | ~g['lon'].between(-5.5, 10)]
resultat('R07', 'latitude 41 à 51,5 et longitude −5,5 à 10', len(g), r07, r07[['livraison_id', 'ville', 'latitude', 'longitude']])

In [ ]:
# R08 · Chaque PDF de facture est rattaché à une commande existante
ids_cmd = set(commandes['commande_id'])
r08 = pdf[~pdf['commande_id'].isin(ids_cmd)]
resultat('R08', 'PDF rattaché à une commande existante', len(pdf), r08, r08[['fichier', 'commande_id', 'taille_octets']])

### 5.3 Au-delà des règles

**À vous** · **logs** : montants vides, types hors référentiel, paiements de commandes inconnues, et écart entre l'heure de réception (`recu`) et `event_time`. Combien d'événements arrivent plus d'une heure après ?

In [ ]:
# Votre code ici
# Piste : hors_referentiel, orphelins, et en_date(recu) − en_date(event_time)

**À vous** · **capteurs** : mesures dans une autre unité, mesures figées (`valeur_figee`), horodatages en double par capteur. Regardez **quand** ont lieu les doublons.

In [ ]:
# Votre code ici
# Piste : capteurs['unite'].value_counts() ; valeur_figee(capteurs, 'capteur_id', 'horodatage_local', 'temperature')

**À vous** · **GPS** : coordonnées inversées, (0, 0), précision perdue (moins de 3 décimales), points à plus de 100 km du centre de leur ville.

In [ ]:
# Votre code ici
# Piste : en_nombre, puis distance_au_centre(livraisons) > 100

**À vous** · **documents** : PDF vides ou sans page, fichiers identiques (même empreinte), commande vide ou inconnue. Ouvrez ensuite deux PDF de `factures_pdf_echantillon/`.

In [ ]:
# Votre code ici
# Piste : doublons(pdf, 'sha256') ; orphelins(pdf, 'commande_id', commandes, 'commande_id')

**À vous** · **texte** : notes hors de 1 à 5, textes vides, encodage cassé (`Ã`, `Â`…), avis en double, produits inexistants. Les avis représentent-ils les clients ?

In [ ]:
# Votre code ici
# Piste : en_nombre(avis['note']).between(1, 5) ; .str.contains('Ã|Â') ; value_counts().sort_index()

### 5.4 Vos règles R09 et R10

Deux anomalies trouvées ci-dessus qu'aucune règle R01 à R08 ne couvre. Écrivez d'abord la règle en français
(périmètre, condition, seuil, action si échec), puis le code, et enregistrez-la avec `resultat`.

In [ ]:
# R09 · (votre règle)
# r09 = ...
# resultat('R09', '...', len(...), r09, r09)

In [ ]:
# R10 · (votre règle)
# r10 = ...
# resultat('R10', '...', len(...), r10, r10)

## 6. Mission 4 · Suivre une commande de bout en bout

`suivre('CMD209')` affiche la commande dans toutes les sources. Choisissez 9 autres commandes (une annulée, une payée en deux fois, une remboursée…).

In [ ]:
suivre('CMD209')

**À vous** · généralisez au moins trois contrôles inter-sources : commande **annulée** mais facturée, payée (sans remboursement) ou livrée ; **livraison avant la commande** ; commande **facturée deux fois**.

In [ ]:
# Votre code ici
# Piste : ensembles de commande_id (set) et isin ; groupby('commande_id').size() > 1

**À vous** · combien de commandes ont **plusieurs** événements de paiement ? Répartissez-les par combinaison de types : lesquelles sont normales ?

In [ ]:
# Votre code ici
# Piste : groupby('commande_id').size() > 1, puis la liste triée des types par commande

## 7. Mission 5 · Synthèse et export

Le plan de remédiation (L3) se rédige dans `modeles/L3_plan_remediation.csv`. Exportez votre synthèse pour la joindre au rapport.

In [ ]:
tableau = pd.DataFrame(synthese).drop_duplicates('règle', keep='last')
tableau.to_csv('synthese_regles.csv', index=False)
tableau

# Bloc 2 · Jour 2 · Mettre sous contrôle

## 8. Mission 6 · Journaux et lineage

In [ ]:
journaux.head(10)

In [ ]:
lineage

**À vous** · journaux : vérifiez `lignes_entree = lignes_chargees + lignes_rejetees`, comptez les statuts par traitement, et repérez la **première date** de chaque série d'échecs.

In [ ]:
# Votre code ici
# Piste : pd.crosstab(traitement, statut) ; filtrer statut != 'SUCCES' et prendre le min des dates

**À vous** · volumes : pour l'import des capteurs, comparez chaque jour au volume attendu (nombre de capteurs × 288 mesures par jour) et à la moyenne des 7 jours précédents. Un statut SUCCES garantit-il des données complètes ?

In [ ]:
# Votre code ici
# Piste : filtrer traitement == 'import_capteurs', puis ecart_a_la_reference(volume par jour)

**À vous** · lineage : pour les flux L03, L04 et L06, confrontez la transformation **déclarée** à ce que vous avez **observé** au Bloc 1.

In [ ]:
# Votre code ici
# Piste : doublons sur event_id ; controle_jointure(commandes, paiements, 'commande_id') ; groupby heure + mean()

## 9. Mission 7 · Données de référence : CRM et ERP

**À vous** · lien ERP → CRM : `client_crm_id` vide ou inconnu. Puis, pour les liens valides, comptez les e-mails et les villes différents, et comparez les dates de mise à jour.

In [ ]:
# Votre code ici
# Piste : orphelins(erp, 'client_crm_id', clients, 'client_id') ; merge puis comparer des colonnes normalisées

**À vous** · retrouvez **Nadia Benali** dans le CRM et l'ERP. D'où viennent les fiches `B-…` ? Combien ont un e-mail identique à une fiche `C…` ?

In [ ]:
# Votre code ici
# Piste : filtrer nom et prénom ; client_id.str.startswith('B-') ; merge sur email

**À vous** · proposez une **règle de rapprochement** : normalisation, paires candidates (blocage), score. Combien de paires propose-t-elle ? Testez aussi la règle « même nom et même prénom » : combien de paires, et combien d'homonymes ?

In [ ]:
# Votre code ici
# Piste : normaliser (majuscules, sans accents), joindre la table à elle-même sur une clé de blocage, puis scorer

## 10. Mission 8 · Dérive de schéma et moniteurs d'observabilité

**À vous** · datez le passage à `schema_version` 2.0 : nombre de commandes par jour et par version en octobre 2026. Rapprochez de `journaux_traitements.csv`.

In [ ]:
# Votre code ici
# Piste : en_date(...).dt.strftime('%Y-%m-%d') puis pd.crosstab(jour, schema_version)

**À vous** · construisez un moniteur de **volume** (commandes par jour vs moyenne des 7 jours précédents) et un moniteur de **schéma** (part des commandes sans `client_id` par jour). Lequel alerte en premier ?

In [ ]:
# Votre code ici
# Piste : ecart_a_la_reference(groupby(jour).size()) et la moyenne de isna() de client_id par jour

## 11. Mission 9 · Indicateurs

In [ ]:
kpi

**À vous** · démontrez la multiplication par la jointure commandes × paiements avec `controle_jointure`, puis chiffrez l'écart sur octobre.

In [ ]:
# Votre code ici
# Piste : controle_jointure(commandes_octobre, paiements, 'commande_id')

**À vous** · recalculez le **chiffre d'affaires d'octobre** selon votre fiche indicateur (HT, commandes uniques, statuts retenus, HT recalculé depuis les lignes, valeurs aberrantes traitées). Comparez aux tableaux A et B.

In [ ]:
# Votre code ici
# Piste : filtrer statuts et dates, recalculer le HT depuis les lignes, agréger par commande

**À vous** · recalculez **clients actifs sur 12 mois** et **conformité de la chaîne du froid** avec vos règles. Expliquez chaque écart avec la valeur publiée.

In [ ]:
# Votre code ici
# Piste : client_id ou customerId, statuts payés, clients existants ; pour le froid : doublons, codes erreur, °F, mesures figées

## Questions pour le dossier

1. R01 est très bas : anomalie de données, de **contrat d'interface** ou de règle ? Quelle action est préventive ?
2. Pour chaque anomalie de votre rapport : **erreur certaine**, **suspicion à valider** ou **cas légitime** ?
3. Quelles sources sont les moins fiables, et sur quels indicateurs chiffrés vous appuyez-vous ?
4. Quel indicateur publieriez-vous tel quel, avec réserve, ou pas du tout ? Pourquoi ?